# KKBox Churn Prediction — Step 2: Feature Engineering
**File:** `02_FeatureEngineering.ipynb`
**Inputs:** clean files from `01_Preprocessing.ipynb`
**Outputs:** `master_model_table.parquet`, `inference_snapshot.parquet` and supporting files
**Next notebook:** `03_Modeling.ipynb`

---

This notebook builds churn labels, computes features, and exports the modeling dataset. All label and feature logic is strictly time-bounded to prevent leakage.

## Data split (Bryan Gregory's setup)

| Split | Month | Users | Churn rate | Labels |
|---|---|---|---|---|
| Train | Jan 2017 | 992,931 | 6.39% | Official train.csv |
| Validation | Feb 2017 | 970,960 | 8.99% | Official train_v2.csv |
| Inference | Mar 2017 | 907,471 | none | sample_submission_v2.csv |

Official competition labels are used directly instead of reconstructing them from the FE pipeline. This guarantees label alignment with competition evaluation criteria.

## Anti-leakage rules

| Component | Data allowed | Data forbidden |
|---|---|---|
| Population | Transactions <= cutoff | Transactions after cutoff |
| Labels | Transactions after last_expire | Not applicable |
| Features | All activity <= cutoff | All activity after cutoff |

## Features (53 total)

| Group | Count | Examples |
|---|---|---|
| Member | 10 | bd, gender, city, days_since_reg, registration_date_abs |
| Transaction | 30 | cancel_rate, auto_renew_rate, days_last_txn_to_expire, recency_to_plan_ratio |
| Log | 13 | total_secs_played, delta_secs_30d_vs_prior, days_since_last_login, has_log_history |

## Note on log coverage

Historical logs cover only 1.1% of train users. However, a value of 0 in log features is itself a valid signal: users with no listening history tend to churn at higher rates. The model learns two separate patterns — one for users with log data and one for users without.


In [19]:

# ===== 1. Imports =====
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import gc

pd.set_option("display.max_columns", 200)

pd.set_option("display.max_rows", 100)

In [20]:

# ===== 2. Paths =====
DATA_DIR = Path("Data")

TX_PATH        = DATA_DIR / "clean_transactions_core.parquet"
MEM_PATH       = DATA_DIR / "clean_members_core.parquet"
LOG_HIST_PATH  = DATA_DIR / "clean_user_logs_hist.parquet"
LOG_MARCH_PATH = DATA_DIR / "clean_user_logs_march.parquet"
META_PATH      = DATA_DIR / "preprocessing_metadata_v3.json"

# Official competition label files (ground-truth)
TRAIN_LABELS_PATH   = DATA_DIR / "train.csv"      # Jan 2017 official labels
TRAIN_V2_LABELS_PATH= DATA_DIR / "train_v2.csv"   # Feb 2017 official labels

# March submission population (official — 907K users)
SUBMISSION_PATH     = DATA_DIR / "sample_submission_v2.csv"

for p in [TX_PATH, MEM_PATH, LOG_HIST_PATH, LOG_MARCH_PATH, META_PATH]:
    print(f"{p}: {'FOUND' if p.exists() else 'MISSING'}")
print()
for p in [TRAIN_LABELS_PATH, TRAIN_V2_LABELS_PATH]:
    print(f"{p}: {'FOUND ✅' if p.exists() else 'MISSING ⚠️  (official labels not available)'}")


Data\clean_transactions_core.parquet: FOUND
Data\clean_members_core.parquet: FOUND
Data\clean_user_logs_hist.parquet: FOUND
Data\clean_user_logs_march.parquet: FOUND
Data\preprocessing_metadata_v3.json: FOUND

Data\train.csv: FOUND ✅
Data\train_v2.csv: FOUND ✅


In [21]:

# ===== 3. Load clean tables =====
if not TX_PATH.exists() or not MEM_PATH.exists():
    raise FileNotFoundError("Thiếu clean core tables. Hãy chạy Preprocessing_timebased_v4/v3 trước.")

transactions    = pd.read_parquet(TX_PATH)
members         = pd.read_parquet(MEM_PATH)
user_logs_hist  = pd.read_parquet(LOG_HIST_PATH)  if LOG_HIST_PATH.exists()  else None
user_logs_march = pd.read_parquet(LOG_MARCH_PATH) if LOG_MARCH_PATH.exists() else None

meta = {}
if META_PATH.exists():
    with open(META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)

# Load official competition labels if available
official_train_labels = None
official_val_labels   = None

if TRAIN_LABELS_PATH.exists():
    official_train_labels = pd.read_csv(TRAIN_LABELS_PATH)
    official_train_labels.columns = [c.lower() for c in official_train_labels.columns]
    official_train_labels["is_churn"] = official_train_labels["is_churn"].astype("int8")
    print(f"official_train_labels (Jan 2017): {official_train_labels.shape}")
    print(f"  churn rate: {official_train_labels['is_churn'].mean():.3f}")
else:
    print("⚠️  train.csv not found — will use FE-derived labels for Jan 2017")

if TRAIN_V2_LABELS_PATH.exists():
    official_val_labels = pd.read_csv(TRAIN_V2_LABELS_PATH)
    official_val_labels.columns = [c.lower() for c in official_val_labels.columns]
    official_val_labels["is_churn"] = official_val_labels["is_churn"].astype("int8")
    print(f"official_val_labels   (Feb 2017): {official_val_labels.shape}")
    print(f"  churn rate: {official_val_labels['is_churn'].mean():.3f}")
else:
    print("⚠️  train_v2.csv not found — will use FE-derived labels for Feb 2017")

# Load official March population from sample_submission_v2.csv
submission_df = None
if SUBMISSION_PATH.exists():
    submission_df = pd.read_csv(SUBMISSION_PATH)
    submission_df.columns = [c.lower() for c in submission_df.columns]
    print(f"\nsubmission_df (March 2017): {submission_df.shape}")
    print(f"  unique users: {submission_df['msno'].nunique():,}")
else:
    print("⚠️  sample_submission_v2.csv not found — March population from FE")

print("\ntransactions:", transactions.shape)
print("members     :", members.shape)
print("user_logs_hist :", None if user_logs_hist is None else user_logs_hist.shape)
print("user_logs_march:", None if user_logs_march is None else user_logs_march.shape)

display(transactions.head())
display(members.head())


official_train_labels (Jan 2017): (992931, 2)
  churn rate: 0.064
official_val_labels   (Feb 2017): (970960, 2)
  churn rate: 0.090

submission_df (March 2017): (907471, 2)
  unique users: 907,471

transactions: (1969768, 11)
members     : (6769473, 12)
user_logs_hist : (106543, 10)
user_logs_march: (396362, 10)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel,expire_gap_days,is_extreme_expiry
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,22,395,1599,1599,0,2016-10-23,2018-02-06,0,471,1
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,41,30,99,99,1,2017-03-15,2017-04-15,0,31,0
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-02-28,2017-04-19,0,50,0
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-03-31,2017-05-19,0,49,0
4,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,41,30,149,149,1,2017-03-26,2017-04-26,0,31,0


,msno,city,bd,gender,registered_via,bd_missing,city_missing,gender_missing,registered_via_missing,registration_year,registration_month,registration_day
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,27.0,Unknown,11,1,0,1,0,2011,9,11
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,27.0,Unknown,7,1,0,1,0,2011,9,14
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,27.0,Unknown,11,1,0,1,0,2011,9,15
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,27.0,Unknown,11,1,0,1,0,2011,9,15
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32.0,female,9,0,0,0,0,2011,9,15


In [22]:

# ===== 4. Setup months =====
# Bryan Gregory's setup: Train=Jan, Val=Feb, Inf=Mar (all 2017)
# Lý do: population Jan/Feb 2017 là active subscribers sắp hết hạn,
# churn rate realistic (~6-15%). Dùng toàn bộ 2015-2016 cho train
# sẽ kéo population là inactive users đã lâu → 99% churn rate giả tạo.
setup_metadata = meta.get("setup_metadata", {})

TRAIN_MONTHS = pd.period_range("2017-01", "2017-01", freq="M")  # Jan 2017 only
VAL_MONTHS   = pd.period_range("2017-02", "2017-02", freq="M")  # Feb 2017 only
INF_MONTH    = pd.Period("2017-03", freq="M")                    # Mar 2017 inference
GRACE_DAYS   = 30

def month_end(period_m):
    return period_m.to_timestamp(how="end").normalize()

inf_cutoff = month_end(INF_MONTH)

print("TRAIN_MONTHS:", TRAIN_MONTHS[0], "->", TRAIN_MONTHS[-1], "| n =", len(TRAIN_MONTHS))
print("VAL_MONTHS  :", VAL_MONTHS[0], "->", VAL_MONTHS[-1], "| n =", len(VAL_MONTHS))
print("INF_MONTH   :", INF_MONTH, "| cutoff =", inf_cutoff)
print("GRACE_DAYS  :", GRACE_DAYS)


TRAIN_MONTHS: 2017-01 -> 2017-01 | n = 1
VAL_MONTHS  : 2017-02 -> 2017-02 | n = 1
INF_MONTH   : 2017-03 | cutoff = 2017-03-31 00:00:00
GRACE_DAYS  : 30


## Label Builder (Vectorized)

### Algorithm

The Scala-aligned method reconstructs each user's effective membership state at the cutoff date rather than filtering raw rows by `membership_expire_date`.

Sort order to determine effective state:
```
transaction_date ASC, is_cancel ASC, membership_expire_date ASC
last row after sort = effective membership state
```

This ordering ensures: the most recent transaction wins, renewals are prioritized over cancellations on the same day, and among the same type, the longer expiry is taken last.

### Six-step logic

| Step | Operation | Constraint |
|---|---|---|
| 1 | Filter transactions <= cutoff | No future data |
| 2 | Sort + groupby().last() | Extract effective state |
| 3 | Filter expire_period == target_month | Select population |
| 4 | Find future transactions after last_expire | No cutoff restriction here |
| 5 | groupby().min() to find earliest renewal | renewal_gap = that date - last_expire |
| 6 | renewal_gap <= 30 days → is_churn = 0 | 30-day grace window |

### Label source classification

| Value | Meaning |
|---|---|
| observed | Renewal data found, label is reliable |
| no_future_data | No transactions found after last_expire |
| beyond_window | last_expire exceeds max transaction date |
| official | Label from competition's official files |


In [23]:
# ===== 5A-7. VECTORIZED label builder =====
# Replaces: _scala_sort_key, _normalize_tx_row, _plan_signature,
#            select_effective_state_by_cutoff, calculate_renewal_gap_scala_like,
#            get_population_users_by_expiry_month, build_labels_from_expiry_month

def build_labels_vectorized(
    transactions: pd.DataFrame,
    target_month: pd.Period,
    grace_days: int = 30,
    debug: bool = False,
):
    """
    Fully vectorized label builder. Equivalent to Scala-aligned state selection
    + renewal gap, but uses pandas groupby instead of per-user Python loops.
    ~100x faster than the row-by-row version.

    Returns DataFrame with columns:
        msno, snapshot_date, last_expire, is_churn, label_source
    """
    cutoff_date = target_month.to_timestamp(how="end").normalize()

    # ── Step 1: history = all rows up to cutoff ───────────────────────────
    hist = transactions[transactions["transaction_date"] <= cutoff_date]

    # ── Step 2: effective state per user via vectorized sort + groupby.last ─
    # Sort order mirrors Scala: tx_date ASC, is_cancel ASC (renewals before cancels
    # on same day), membership_expire_date ASC (longer expiry last among renewals)
    # → last row per user = effective state
    hist_sorted = hist.sort_values(
        ["msno", "transaction_date", "is_cancel", "membership_expire_date"],
        ascending=[True, True, True, True],
    )
    effective = hist_sorted.groupby("msno", sort=False).last().reset_index()

    # ── Step 3: population = users whose effective expiry is in target month ─
    effective["expire_period"] = effective["membership_expire_date"].dt.to_period("M")
    population = (
        effective[effective["expire_period"] == target_month]
        [["msno", "membership_expire_date"]]
        .rename(columns={"membership_expire_date": "last_expire"})
        .copy()
    )

    if len(population) == 0:
        if debug:
            print(f"[{target_month}] No population found — returning empty DataFrame")
        return pd.DataFrame(
            columns=["msno", "snapshot_date", "last_expire", "is_churn", "label_source"]
        )

    # ── Step 4: future non-cancel transactions after each user's last_expire ─
    # NOTE: no > cutoff_date pre-filter here — cutoff only constrains FEATURES.
    # For labels, the renewal window starts at last_expire (per user), not month-end.
    # The per-user filter on line below correctly scopes each user's grace window.
    future = transactions[
        transactions["is_cancel"] != 1
    ].copy()
    future = future.merge(population[["msno", "last_expire"]], on="msno", how="inner")
    future = future[future["transaction_date"] > future["last_expire"]]

    # ── Step 5: first renewal date per user → renewal gap ────────────────
    if len(future) > 0:
        first_renewal = (
            future.groupby("msno")["transaction_date"]
            .min()
            .reset_index()
            .rename(columns={"transaction_date": "first_renewal_date"})
        )
        population = population.merge(first_renewal, on="msno", how="left")
        population["renewal_gap"] = (
            population["first_renewal_date"] - population["last_expire"]
        ).dt.days
    else:
        population["first_renewal_date"] = pd.NaT
        population["renewal_gap"] = np.nan

    # ── Step 6: assign labels ─────────────────────────────────────────────
    max_txn_date = transactions["transaction_date"].max()

    population["is_churn"]      = np.int8(1)
    population["label_source"]  = "no_future_data"

    observed_mask = population["renewal_gap"].notna()
    population.loc[observed_mask, "label_source"] = "observed"
    population.loc[
        observed_mask & (population["renewal_gap"] <= grace_days), "is_churn"
    ] = np.int8(0)

    beyond_mask = population["last_expire"] > max_txn_date
    population.loc[beyond_mask, "label_source"] = "beyond_window"
    population.loc[beyond_mask, "is_churn"]     = np.int8(1)

    population["snapshot_date"] = cutoff_date

    result = population[
        ["msno", "snapshot_date", "last_expire", "is_churn", "label_source"]
    ].copy()

    if debug:
        n          = len(result)
        churn_rate = result["is_churn"].mean()
        observed   = (result["label_source"] == "observed").sum()
        print(
            f"[{target_month}] population={n:,}  churn_rate={churn_rate:.3f}  "
            f"observed={observed:,}  no_future={n - observed:,}"
        )

    return result


In [24]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


In [25]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


## Population and Label Builder

Population and labels are built together in a single call to `build_labels_vectorized`. No separate population loop is needed.

Population for a given month = users whose effective membership state (reconstructed at the cutoff) has a `membership_expire_date` falling in that month.

In the actual pipeline, population is taken directly from the official label files (`train.csv`, `train_v2.csv`) rather than from FE reconstruction. This eliminates any mismatch between FE-derived and competition-defined populations.


In [26]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


## Churn Labels

See `build_labels_vectorized`. Label formula:

```
is_churn = 0  if  renewal_gap <= 30 days  (renewed on time)
is_churn = 1  if  renewal_gap > 30 days   (late renewal or no renewal)
is_churn = 1  if  no renewal transaction found
```

Important: `renewal_gap` is computed from each user's `last_expire`, not from the month-end cutoff. The cutoff only determines effective state and feature computation — it does not restrict the renewal search window.


In [27]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


In [28]:
# ===== Pre-group transactions (kept for reference only) =====
# transactions_grouped is no longer needed by build_labels_vectorized.
# build_core_feature_snapshot still takes the full transactions DataFrame directly.
# This cell can be skipped.

# Compute global max transaction date (still useful for reference)
MAX_TXN_DATE_GLOBAL = transactions["transaction_date"].max()
print(f"Global max transaction date: {MAX_TXN_DATE_GLOBAL}")

PIPELINE_START_TIME = time.time()


Global max transaction date: 2017-03-31 00:00:00


In [29]:
# ===== LOG COVERAGE DIAGNOSTIC =====
# Run this to understand how many train/val users have log data.
# If coverage < 20%, log features will be mostly zero and add noise.

if user_logs_hist is not None and official_train_labels is not None:
    train_pop  = set(official_train_labels["msno"].astype(str))
    log_users  = set(user_logs_hist["msno"].astype(str))
    overlap    = train_pop & log_users
    coverage   = len(overlap) / len(train_pop)
    print(f"Train population (Jan 2017)    : {len(train_pop):,}")
    print(f"user_logs_hist unique users    : {len(log_users):,}")
    print(f"Overlap                        : {len(overlap):,}")
    print(f"Log coverage of train pop      : {coverage:.1%}")

    if official_val_labels is not None:
        val_pop     = set(official_val_labels["msno"].astype(str))
        val_overlap = val_pop & log_users
        val_cov     = len(val_overlap) / len(val_pop)
        print(f"\nVal population (Feb 2017)      : {len(val_pop):,}")
        print(f"Log coverage of val pop        : {val_cov:.1%}")

    print(f"\nuser_logs_hist columns: {user_logs_hist.columns.tolist()}")
    print(f"user_logs_hist date range: "
          f"{user_logs_hist['date'].min()} → {user_logs_hist['date'].max()}")

    if coverage < 0.20:
        print(f"\n⚠️  Coverage {coverage:.1%} < 20% — log features will be mostly 0")
        print("   Consider disabling logs_df in build_core_feature_snapshot")
        print("   Set logs_df=None in Cell 18 and Cell 20 to test without logs")
    elif coverage >= 0.50:
        print(f"\n✅ Coverage {coverage:.1%} — log features will add genuine signal")
    else:
        print(f"\n⚠️  Coverage {coverage:.1%} — partial signal, worth keeping but noisy")
else:
    print("⚠️  user_logs_hist or official_train_labels not loaded — skipping coverage check")

# Also check user_logs_march coverage (March logs, 316K users)
if user_logs_march is not None and official_train_labels is not None:
    march_log_users = set(user_logs_march["msno"].astype(str))
    train_pop_set   = set(official_train_labels["msno"].astype(str))
    march_overlap   = train_pop_set & march_log_users
    march_cov       = len(march_overlap) / len(train_pop_set)
    print(f"\nuser_logs_march (316K users) coverage of train pop: {march_cov:.1%}")
    print(f"  Overlap: {len(march_overlap):,} users")
    print("  Note: march logs cover 2017-03 only — different time period than train")
    print("        but user identities may still overlap → signal for same users")

# Combined coverage (hist + march together)
if user_logs_hist is not None and user_logs_march is not None and official_train_labels is not None:
    combined_log_users = (
        set(user_logs_hist["msno"].astype(str)) |
        set(user_logs_march["msno"].astype(str))
    )
    train_pop_set = set(official_train_labels["msno"].astype(str))
    combined_overlap = train_pop_set & combined_log_users
    combined_cov     = len(combined_overlap) / len(train_pop_set)
    print(f"\nCombined logs (hist + march) coverage of train: {combined_cov:.1%}")
    print(f"  Combined log users: {len(combined_log_users):,}")
    print(f"  Overlap with train: {len(combined_overlap):,}")


Train population (Jan 2017)    : 992,931
user_logs_hist unique users    : 22,443
Overlap                        : 10,708
Log coverage of train pop      : 1.1%

Val population (Feb 2017)      : 970,960
Log coverage of val pop        : 1.1%

user_logs_hist columns: ['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs', 'total_plays_proxy']
user_logs_hist date range: 2015-01-01 00:00:00 → 2017-02-28 00:00:00

⚠️  Coverage 1.1% < 20% — log features will be mostly 0
   Consider disabling logs_df in build_core_feature_snapshot
   Set logs_df=None in Cell 18 and Cell 20 to test without logs

user_logs_march (316K users) coverage of train pop: 22.9%
  Overlap: 227,742 users
  Note: march logs cover 2017-03 only — different time period than train
        but user identities may still overlap → signal for same users

Combined logs (hist + march) coverage of train: 23.7%
  Combined log users: 334,707
  Overlap with train: 235,363


In [30]:
# ===== 8. Runtest labels (FAST DEBUG — vectorized) =====
sample_train_month = pd.Period("2016-12", freq="M")
sample_val_month   = pd.Period("2017-01", freq="M")

# Stratified sample — guarantee users with expiries in target months are included
target_periods = [pd.Period(m, freq="M") for m in
                  ["2016-10", "2016-11", "2016-12", "2017-01", "2017-02"]]
target_msnos = transactions[
    transactions["membership_expire_date"].dt.to_period("M").isin(target_periods)
]["msno"].unique()
other_msnos = (
    transactions[~transactions["msno"].isin(target_msnos)]["msno"]
    .drop_duplicates()
    .sample(max(0, 50000 - len(target_msnos)), random_state=42)
)
debug_users = list(target_msnos) + list(other_msnos)
transactions_debug = transactions[transactions["msno"].isin(debug_users)].copy()

print(f"Debug sample: {len(target_msnos):,} target-month users + "
      f"{len(other_msnos):,} filler = {len(debug_users):,} total")
print(f"Debug transactions rows: {len(transactions_debug):,}")

print("\n" + "="*80)
start_train = time.time()
sample_train_labels = build_labels_vectorized(
    transactions_debug, sample_train_month, grace_days=GRACE_DAYS, debug=True
)
train_time = time.time() - start_train

print("="*80)
start_val = time.time()
sample_val_labels = build_labels_vectorized(
    transactions_debug, sample_val_month, grace_days=GRACE_DAYS, debug=True
)
val_time = time.time() - start_val

print("="*80)
print(f"sample_train_labels shape : {sample_train_labels.shape} ({train_time:.2f}s)")
print(f"sample_val_labels shape   : {sample_val_labels.shape} ({val_time:.2f}s)")
print(f"\nLabel source breakdown (train):")
print(sample_train_labels["label_source"].value_counts())

# ── Corrected time estimate ──────────────────────────────────────────────
n_debug_total     = transactions_debug["msno"].nunique()
n_full_total      = transactions["msno"].nunique()
scale             = n_full_total / max(n_debug_total, 1)
time_per_user     = (train_time + val_time) / max(n_debug_total, 1)
avg_pop           = (len(sample_train_labels) + len(sample_val_labels)) / 2
est_pop_full      = avg_pop * scale
est_per_month     = time_per_user * est_pop_full
n_months_total    = len(TRAIN_MONTHS) + len(VAL_MONTHS) + 1

print(f"\nCORRECTED TIME ESTIMATE (vectorized)")
print(f"  debug sample size         : {n_debug_total:,} users")
print(f"  debug time (2 months)     : {train_time + val_time:.2f}s")
print(f"  est. population/month     : ~{est_pop_full:,.0f} users")
print(f"  est. time/month (full)    : ~{est_per_month:.1f}s")
print(f"  est. total ({n_months_total} months)     : "
      f"~{est_per_month * n_months_total:.0f}s "
      f"(~{est_per_month * n_months_total / 60:.1f} min)")


Debug sample: 127,970 target-month users + 0 filler = 127,970 total
Debug transactions rows: 310,923

[2016-12] population=25,177  churn_rate=0.994  observed=21,905  no_future=3,272
[2017-01] population=24,625  churn_rate=0.975  observed=22,026  no_future=2,599
sample_train_labels shape : (25177, 5) (0.25s)
sample_val_labels shape   : (24625, 5) (0.26s)

Label source breakdown (train):
label_source
observed          21905
no_future_data     3272
Name: count, dtype: int64

CORRECTED TIME ESTIMATE (vectorized)
  debug sample size         : 127,970 users
  debug time (2 months)     : 0.51s
  est. population/month     : ~255,361 users
  est. time/month (full)    : ~1.0s
  est. total (3 months)     : ~3s (~0.1 min)


## Feature Engineering (Bryan-aligned)

### Two temporal methods from Bryan's paper

Bryan uses two distinct approaches for date-based features. Using the wrong method for a given feature introduces bias.

| Method | How it works | Example |
|---|---|---|
| Relative Refactoring | Convert date to days elapsed since prediction period start | `days_since_reg`, `days_since_last_txn` |
| Absolute Method | Keep date as raw YYYYMMDD integer | `registration_date_abs` |

The relative method ensures features are comparable across different time periods (train vs test). The absolute method is used when calendar-date patterns carry more signal than elapsed time — for example, if users who registered after a major holiday have distinct churn behavior.

### Full feature list (53 features after FE)

Member features (10):

| Feature | Method | Description |
|---|---|---|
| `bd`, `bd_missing` | — | Age and missing flag |
| `city`, `city_missing` | — | City and missing flag |
| `gender`, `gender_missing` | — | Gender and missing flag |
| `registered_via`, `registered_via_missing` | — | Registration channel and missing flag |
| `days_since_reg` | Relative | Days from registration to cutoff |
| `registration_date_abs` | Absolute | YYYYMMDD integer — captures calendar registration patterns |

Transaction features (30):

| Feature | Bryan reference | Description |
|---|---|---|
| `n_txns` | — | Total transaction count |
| `auto_renew_rate` | — | Fraction of transactions with auto-renew |
| `last_is_auto_renew` | — | Auto-renew flag on most recent transaction |
| `avg_plan_days`, `plan_days_std` | — | Mean and std of subscription plan length |
| `last_plan_days`, `first_plan_days`, `max_plan_days` | Bryan #1 (6.61%) | Plan length variations |
| `plan_change_flag` | — | Whether user ever changed plan |
| `total_amount_paid`, `avg_amount_paid`, `max_amount_paid` | — | Payment amount statistics |
| `zero_paid_rate`, `avg_discount_rate` | — | Promo usage signals |
| `share_30d`, `share_90d` | Relative | Fraction of transactions in last 30/90 days |
| `days_since_last_txn` | Relative | Days since most recent transaction |
| `tenure_days` | Relative | Account age |
| `cancel_rate`, `cancel_count` | Bryan paper | Historical cancellation rate and count |
| `last_is_cancel` | Bryan paper | Was the most recent transaction a cancellation |
| `cancel_in_last_month` | Bryan paper | Cancellation occurred in last 30 days |
| `avg_cancel_per_month` | Bryan paper | Average cancellations per calendar month |
| `cancel_to_plandays_ratio` | Bryan #6 (3.81%) | Cancellation count / total plan days |
| `active_no_cancel_flag` | Bryan #9 (2.80%) | More than 1 transaction in last month AND no cancellation |
| `days_since_last_cancel` | Bryan paper | Days since most recent cancellation |
| `payment_method_nunique` | — | Number of distinct payment methods used |
| `days_last_txn_to_expire` | Relative | Days between last transaction and expiry |
| `recency_to_plan_ratio` | Higher-order | days_since_last_txn / avg_plan_days |
| `spend_rate_per_day` | Higher-order | total_amount_paid / tenure_days |

Log features (13):

| Feature | Bryan reference | Description |
|---|---|---|
| `total_secs_played` | Bryan #3 equivalent | Total seconds played across all history |
| `total_uniq_songs` | Bryan #3 (4.19%) | Total unique songs played |
| `num_logins` | Bryan #8 (2.95%) | Total login days in history |
| `secs_played_14d`, `logins_14d` | Bryan #2 window | Activity in last 14 days |
| `secs_played_30d`, `logins_30d` | Prior month | Activity in last 30 days |
| `logins_90d` | — | Login count in last 90 days |
| `secs_played_60_30d` | Bryan #5 (3.98%) | Seconds played in prior-prior month (30-60 days ago) |
| `delta_secs_30d_vs_prior` | Bryan #4 (3.99%) | Trend: avg secs/day change from 60-30d to last 30d |
| `delta_songs_14d_vs_30d` | Bryan #2 (4.56%) | Trend: avg unique songs/day change, last 14d vs last 30d |
| `days_since_last_login` | Bryan #10 (2.63%) | Days since most recent login |
| `max_secs_single_day` | Bryan max aggregation | Peak listening in a single day |
| `days_since_significant_usage` | Bryan paper | Days since last session with meaningful listening |
| `has_log_history` | Derived | Binary flag: 1 if user has any log records, 0 otherwise |

Note: the run that produced the current parquet files used `num_uniq_songs` (wrong column name). The fix to `num_uniq` has been applied to the notebook. Song-based features (`total_uniq_songs`, `delta_songs_14d_vs_30d`, `uniq_songs_std_30d`) will populate correctly on the next FE run.


In [31]:

# ===== 8A. build_core_feature_snapshot — Bryan-aligned =====
# Implements both Relative Refactoring and Absolute temporal methods.
# Total features: ~40 (transaction + member + higher-order)
def build_core_feature_snapshot(
    cutoff_date,
    transactions_df,
    members_df,
    population_users=None,
    logs_df=None,
):
    cutoff_date = pd.to_datetime(cutoff_date)

    tx = transactions_df[pd.to_datetime(transactions_df["transaction_date"], errors="coerce") <= cutoff_date].copy()
    mem = members_df.copy()

    if population_users is not None:
        tx  = tx[tx["msno"].astype("string").isin(population_users)].copy()
        mem = mem[mem["msno"].astype("string").isin(population_users)].copy()

    # ── MEMBER FEATURES ───────────────────────────────────────────────────────
    mem_feat = mem.copy()

    # Relative Refactoring: days_since_reg (days from registration to prediction period start)
    if "registration_year" in mem_feat.columns and "registration_month" in mem_feat.columns:
        reg_date = pd.to_datetime(
            mem_feat["registration_year"].astype(str) + "-" +
            mem_feat["registration_month"].astype(str).str.zfill(2) + "-01",
            errors="coerce"
        )
        mem_feat["days_since_reg"] = (cutoff_date - reg_date).dt.days.clip(lower=0)

        # Absolute Method (Bryan Slide 4): raw registration date as YYYYMMDD integer
        # Useful when calendar-date churn patterns exist (e.g. holiday registrations)
        # 02-4 FIX: include day component if preserved from 01_Preprocessing
        # registration_init_time was decomposed into year/month/day in 01.
        # If day column is available, use it; otherwise fall back to 1 (mid-precision).
        reg_year  = mem_feat["registration_year"].fillna(0).astype(int)
        reg_month = mem_feat["registration_month"].fillna(1).astype(int)
        if "registration_day" in mem_feat.columns:
            reg_day = mem_feat["registration_day"].fillna(1).astype(int)
        else:
            reg_day = 1
            print("  Warning: registration_day not found — registration_date_abs day precision lost")
        mem_feat["registration_date_abs"] = (
            reg_year * 10000 + reg_month * 100 + reg_day
        ).astype(float)

    mem_keep = ["msno"] + [c for c in [
        "bd", "bd_missing",
        "city", "city_missing",
        "gender", "gender_missing",
        "registered_via", "registered_via_missing",
        "days_since_reg",
        "registration_date_abs",   # ← Absolute method
    ] if c in mem_feat.columns]
    mem_feat = mem_feat[mem_keep].copy()

    # ── TRANSACTION FEATURES ─────────────────────────────────────────────────
    tx_feat = pd.DataFrame(columns=["msno"])
    if len(tx) > 0:
        tx = tx.sort_values(["msno", "transaction_date", "membership_expire_date"])
        grp = tx.groupby("msno", dropna=False)

        tx_feat = pd.DataFrame({"msno": list(grp.size().index)})
        tx_feat["n_txns"] = grp.size().values

        # ── Auto-renew ────────────────────────────────────────────────────────
        if "is_auto_renew" in tx.columns:
            tx_feat["auto_renew_rate"]    = grp["is_auto_renew"].mean().values
            # Last transaction auto-renew: current user intent (stronger than average)
            tx_feat["last_is_auto_renew"] = grp["is_auto_renew"].last().values

        # ── Plan features ─────────────────────────────────────────────────────
        if "payment_plan_days" in tx.columns:
            tx_feat["avg_plan_days"]    = grp["payment_plan_days"].mean().values
            tx_feat["plan_days_std"]    = grp["payment_plan_days"].std().fillna(0).values
            tx_feat["plan_change_flag"] = (grp["payment_plan_days"].nunique() > 1).astype("int8").values
            # Last plan days: most recent subscription length chosen (Bryan #1 feature)
            tx_feat["last_plan_days"]   = grp["payment_plan_days"].last().values
            # First plan days: original subscription plan (cohort signal)
            tx_feat["first_plan_days"]  = grp["payment_plan_days"].first().values
            # Max plan days ever chosen (loyalty signal)
            tx_feat["max_plan_days"]    = grp["payment_plan_days"].max().values

        # ── Amount features ───────────────────────────────────────────────────
        if "actual_amount_paid" in tx.columns:
            tx_feat["total_amount_paid"] = grp["actual_amount_paid"].sum().values
            tx_feat["avg_amount_paid"]   = grp["actual_amount_paid"].mean().values
            tx_feat["max_amount_paid"]   = grp["actual_amount_paid"].max().values
            tx_feat["zero_paid_rate"]    = grp["actual_amount_paid"].apply(lambda s: (s == 0).mean()).values

        if all(c in tx.columns for c in ["plan_list_price", "actual_amount_paid"]):
            price_mean = grp["plan_list_price"].mean()
            paid_mean  = grp["actual_amount_paid"].mean()
            tx_feat["avg_discount_rate"] = ((price_mean - paid_mean) / price_mean.replace(0, np.nan)).values
            tx_feat["avg_discount_rate"] = tx_feat["avg_discount_rate"].replace([np.inf, -np.inf], np.nan)

        # ── Recency shares ────────────────────────────────────────────────────
        tx_date_col  = pd.to_datetime(tx["transaction_date"], errors="coerce")
        total_cnt    = grp.size()

        tx_recent_30 = tx[tx_date_col > (cutoff_date - pd.Timedelta(days=30))]
        grp_30       = tx_recent_30.groupby("msno").size()
        tx_feat["share_30d"] = (grp_30 / total_cnt).reindex(total_cnt.index).fillna(0).values

        tx_recent_90 = tx[tx_date_col > (cutoff_date - pd.Timedelta(days=90))]
        grp_90       = tx_recent_90.groupby("msno").size()
        tx_feat["share_90d"] = (grp_90 / total_cnt).reindex(total_cnt.index).fillna(0).values

        # ── Recency / tenure ──────────────────────────────────────────────────
        last_txn  = pd.to_datetime(grp["transaction_date"].max(), errors="coerce")
        first_txn = pd.to_datetime(grp["transaction_date"].min(), errors="coerce")
        tx_feat["days_since_last_txn"] = (cutoff_date - last_txn).dt.days.values
        tx_feat["tenure_days"]         = (cutoff_date - first_txn).dt.days.values

        # ── Cancellation features (Bryan Slide 1: key transaction signals) ────
        if "is_cancel" in tx.columns:
            tx_feat["cancel_rate"]  = grp["is_cancel"].mean().values
            tx_feat["cancel_count"] = grp["is_cancel"].sum().values

            # Last transaction is a cancellation (strongest churn signal)
            tx_feat["last_is_cancel"] = grp["is_cancel"].last().astype("int8").values

            # Cancel in last 30 days (boolean flag — Bryan's prior_month_cancel)
            tx_cancel_30 = tx[
                (tx_date_col > (cutoff_date - pd.Timedelta(days=30))) &
                (tx["is_cancel"] == 1)
            ]
            cancel_30_flag = tx_cancel_30.groupby("msno").size().reindex(
                total_cnt.index, fill_value=0
            ) > 0
            tx_feat["cancel_in_last_month"] = cancel_30_flag.astype("int8").values

            # Avg cancellations per month (Bryan: avg_cancel_rate_per_month)
            tenure_months = (tx_feat["tenure_days"] / 30.0).clip(lower=1)
            tx_feat["avg_cancel_per_month"] = tx_feat["cancel_count"] / tenure_months

            # Bryan Top10 #6: ratio of cancellations to total plan days
            # (cancel_count / sum_of_all_plan_days — different from avg_plan_days)
            if "payment_plan_days" in tx.columns:
                total_plan_days = grp["payment_plan_days"].sum()
                tx_feat["cancel_to_plandays_ratio"] = (
                    tx_feat["cancel_count"] / total_plan_days.values.clip(min=1)
                )

            # Bryan Top10 #9: flag if >1 tx in prior month AND no cancellations
            tx_30_size = tx_recent_30.groupby("msno").size().reindex(
                total_cnt.index, fill_value=0
            )
            active_no_cancel = ((tx_30_size > 1) & (~cancel_30_flag)).astype("int8")
            tx_feat["active_no_cancel_flag"] = active_no_cancel.values

            # days_since_last_cancellation (relative refactoring method)
            # Paper explicitly mentions this as a key temporal feature
            cancel_tx = tx[tx["is_cancel"] == 1].copy()
            # 02-3 FIX: users who never cancelled get sentinel 9999 (not 0)
            # 0 would be indistinguishable from "cancelled today".
            # also add has_ever_cancelled flag for explicit model signal.
            NEVER_CANCELLED_SENTINEL = 9999
            if len(cancel_tx) > 0:
                last_cancel_date = (
                    pd.to_datetime(cancel_tx["transaction_date"], errors="coerce")
                    .groupby(cancel_tx["msno"]).max()
                )
                days_since_cancel = (cutoff_date - last_cancel_date).dt.days
                tx_feat["days_since_last_cancel"] = (
                    days_since_cancel.reindex(grp.size().index,
                                             fill_value=NEVER_CANCELLED_SENTINEL)
                    .clip(lower=0).values
                )
                tx_feat["has_ever_cancelled"] = (
                    days_since_cancel.reindex(grp.size().index, fill_value=-1) >= 0
                ).astype("int8").values
            else:
                tx_feat["days_since_last_cancel"] = NEVER_CANCELLED_SENTINEL
                tx_feat["has_ever_cancelled"] = 0

        if "payment_method_id" in tx.columns:
            tx_feat["payment_method_nunique"] = grp["payment_method_id"].nunique().values

        # ── Expiry gap feature ────────────────────────────────────────────────
        if "membership_expire_date" in tx.columns:
            last_expire_per_user = pd.to_datetime(
                grp["membership_expire_date"].max(), errors="coerce"
            )
            tx_feat["days_last_txn_to_expire"] = (
                last_expire_per_user - last_txn
            ).dt.days.values

        # ── Higher-order / ratio features (Bryan Slide 1: interaction features) ──
        # Ratio of days_since_last_txn to avg_plan_days
        # Tells the model: is the user overdue relative to their typical plan length?
        tx_feat["recency_to_plan_ratio"] = (
            tx_feat["days_since_last_txn"] /
            tx_feat["avg_plan_days"].replace(0, np.nan)
        )
        # Spend rate: total money paid per day of tenure
        tx_feat["spend_rate_per_day"] = (
            tx_feat["total_amount_paid"] /
            tx_feat["tenure_days"].replace(0, np.nan)
        )

    # ── USER LOGS FEATURES (optional — auxiliary branch) ─────────────────────
    log_feat = pd.DataFrame(columns=["msno"])
    if logs_df is not None and len(logs_df) > 0:
        logs = logs_df.copy()
        logs["date"] = pd.to_datetime(logs["date"], errors="coerce")
        logs = logs[logs["date"] <= cutoff_date]
        if population_users is not None:
            logs = logs[logs["msno"].astype("string").isin(population_users)]

        if len(logs) > 0:
            lg = logs.groupby("msno")
            log_feat = pd.DataFrame({"msno": list(lg.size().index)})

            # Total behavioral signals (all history)
            if "total_secs" in logs.columns:
                log_feat["total_secs_played"] = lg["total_secs"].sum().values
            if "num_uniq" in logs.columns:
                log_feat["total_uniq_songs"]  = lg["num_uniq"].sum().values

            # Number of login days (each log row = 1 login day)
            log_feat["num_logins"] = lg.size().values

            # Last 14 days (Bryan's #2 feature uses this window)
            logs_14 = logs[logs["date"] > (cutoff_date - pd.Timedelta(days=14))]
            lg14    = logs_14.groupby("msno")

            if "num_uniq" in logs.columns:
                log_feat["uniq_songs_14d"] = (
                    lg14["num_uniq"].sum()
                    .reindex(lg.size().index, fill_value=0).values
                )
            if "total_secs" in logs.columns:
                log_feat["secs_played_14d"] = (
                    lg14["total_secs"].sum()
                    .reindex(lg.size().index, fill_value=0).values
                )
            log_feat["logins_14d"] = (
                lg14.size().reindex(lg.size().index, fill_value=0).values
            )

            # Last 30 days behavioral signals
            logs_30 = logs[logs["date"] > (cutoff_date - pd.Timedelta(days=30))]
            lg30    = logs_30.groupby("msno")

            if "total_secs" in logs.columns:
                log_feat["secs_played_30d"] = (
                    lg30["total_secs"].sum()
                    .reindex(lg.size().index, fill_value=0).values
                )
            log_feat["logins_30d"] = (
                lg30.size().reindex(lg.size().index, fill_value=0).values
            )

            # Last 90 days
            logs_90 = logs[logs["date"] > (cutoff_date - pd.Timedelta(days=90))]
            lg90    = logs_90.groupby("msno")
            log_feat["logins_90d"] = (
                lg90.size().reindex(lg.size().index, fill_value=0).values
            )

            # Prior-prior month window: 30-60 days ago
            logs_60_30 = logs[
                (logs["date"] > (cutoff_date - pd.Timedelta(days=60))) &
                (logs["date"] <= (cutoff_date - pd.Timedelta(days=30)))
            ]
            lg60_30 = logs_60_30.groupby("msno")

            # Bryan Top10 #5: secs in prior-prior month
            if "total_secs" in logs.columns:
                secs_60_30 = lg60_30["total_secs"].sum().reindex(
                    lg.size().index, fill_value=0
                )
                log_feat["secs_played_60_30d"] = secs_60_30.values

                # Bryan Top10 #4: TREND — change in avg secs/day (30d vs 60_30d)
                secs_30d_val  = lg30["total_secs"].sum().reindex(lg.size().index, fill_value=0)
                days_in_30    = 30.0
                log_feat["delta_secs_30d_vs_prior"] = (
                    (secs_30d_val / days_in_30) - (secs_60_30 / days_in_30)
                ).values

            if "num_uniq" in logs.columns:
                songs_30d_val = lg30["num_uniq"].sum().reindex(lg.size().index, fill_value=0)
                songs_60_30   = lg60_30["num_uniq"].sum().reindex(lg.size().index, fill_value=0)

                # Bryan Top10 #2: TREND — change in avg unique songs/day
                # 30d vs 60_30d
                log_feat["delta_songs_30d_vs_prior"] = (
                    (songs_30d_val / 30.0) - (songs_60_30 / 30.0)
                ).values

                # ul_lastmo_last2wk_numunq_avg_diff: Bryan's EXACT #2 feature
                # avg numunq per day in last 14 days vs avg per day in prior month
                songs_14d_val = lg14["num_uniq"].sum().reindex(lg.size().index, fill_value=0)
                log_feat["delta_songs_14d_vs_30d"] = (
                    (songs_14d_val / 14.0) - (songs_30d_val / 30.0)
                ).values

                # ul_mo1_mo2_trend for numunq: trend month-over-month
                log_feat["delta_songs_mo1_mo2"] = (
                    (songs_30d_val / 30.0) - (songs_60_30 / 30.0)
                ).values  # same as delta_songs_30d_vs_prior but aliased for clarity

                # Bryan Top10 #7: std dev of unique songs in prior month
                log_feat["uniq_songs_std_30d"] = (
                    logs_30.groupby("msno")["num_uniq"].std()
                    .reindex(lg.size().index, fill_value=0).values
                )

            # Relative Refactoring: days_since_last_login
            last_login = lg["date"].max()
            log_feat["days_since_last_login"] = (
                cutoff_date - last_login
            ).dt.days.values

            # MAX aggregations (Bryan used sum/mean/max/std)
            if "total_secs" in logs.columns:
                log_feat["max_secs_single_day"] = lg["total_secs"].max().reindex(
                    lg.size().index, fill_value=0
                ).values

            if "num_uniq" in logs.columns:
                log_feat["max_uniq_single_day"] = lg["num_uniq"].max().reindex(
                    lg.size().index, fill_value=0
                ).values

            # days_since_last_significant_usage (paper mentions explicitly)
            # Significant = session with total_secs > 1000 (roughly 16 min)
            if "total_secs" in logs.columns:
                significant_threshold = 1000
                sig_logs = logs[logs["total_secs"] > significant_threshold]
                if len(sig_logs) > 0:
                    last_sig = sig_logs.groupby("msno")["date"].max()
                    log_feat["days_since_significant_usage"] = (
                        (cutoff_date - last_sig)
                        .dt.days
                        .reindex(lg.size().index, fill_value=0)
                        .clip(lower=0).values
                    )
                else:
                    log_feat["days_since_significant_usage"] = 0

    # ── ASSEMBLE OUTPUT ───────────────────────────────────────────────────────
    user_pool = pd.Series(dtype="string")
    for part in [mem_feat, tx_feat, log_feat]:
        if "msno" in part.columns:
            user_pool = pd.concat([user_pool, part["msno"].astype("string")], ignore_index=True)

    out = pd.DataFrame({"msno": user_pool.dropna().drop_duplicates()})
    out["snapshot_date"] = cutoff_date

    for part in [mem_feat, tx_feat, log_feat]:
        if "msno" in part.columns:
            out = out.merge(part, on="msno", how="left")

    # ── Final type enforcement + NaN/inf cleanup ──────────────────────────────
    # has_log_history: explicit binary flag for log-absent users (98.9%)
    # 0 = no listening data in dataset → itself a churn signal
    # 1 = has log records → engagement signal
    if len(log_feat) > 0 and "msno" in log_feat.columns:
        log_users_set = set(log_feat["msno"].astype(str))
    else:
        log_users_set = set()
    out["has_log_history"] = out["msno"].astype(str).isin(log_users_set).astype("int8")

    SKIP_COLS = {"msno", "snapshot_date"}
    for c in out.columns:
        if c in SKIP_COLS:
            continue
        dtype_str = str(out[c].dtype)
        if "string" in dtype_str.lower() or dtype_str in ["object", "category"]:
            out[c] = out[c].astype(object).fillna("Unknown").astype(str)
        else:
            out[c] = pd.to_numeric(out[c], errors="coerce")
            out[c] = out[c].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Sanity assertion
    num_out = out.select_dtypes(include="number")
    nan_rem = num_out.isna().sum().sum()
    inf_rem = np.isinf(num_out).sum().sum()
    if nan_rem > 0 or inf_rem > 0:
        import warnings
        warnings.warn(f"[build_core_feature_snapshot] "
                      f"{nan_rem} NaN + {inf_rem} inf remain after cleanup.")

    return out


## Build Train and Validation Snapshots

Each snapshot is built in sequence: get labels → determine population → compute features → merge.

```
1. build_snapshot_with_official_labels(month)
   |-- If official labels available: use train.csv / train_v2.csv
   |-- Else: fall back to build_labels_vectorized
2. build_core_feature_snapshot(cutoff, transactions, members, logs)
   |-- Features computed on history <= cutoff (no leakage)
   |-- Log features added if USE_LOGS = True
3. merge labels + features on msno and snapshot_date
```

Jan 2017 features use all history up to 2017-01-31. Feb 2017 features use all history up to 2017-02-28, which includes January data. Validation therefore has one additional month of behavioral history compared to training.


In [32]:
# ===== 9. Build Train + Validation (Bryan-style + Official Labels) =====
# USE_LOGS: set True only if log coverage >= 20% (see diagnostic cell above)
# If coverage is low, logs add zero-inflated noise and hurt model performance
# Log coverage = 1.1% (10,708 / 992,931 Jan 2017 users have log history)
# 0-value IS valid signal: 'no recorded listening history' → higher churn risk
# The model learns separately for log-present vs log-absent users.
# Bryan's log features show 2-4% importance each despite low coverage.
USE_LOGS = True   # Keep True — zeros carry signal, not pure noise


# Strategy:
#   Train : Jan 2017 population — labels from official train.csv if available,
#           else fallback to FE-derived labels
#   Val   : Feb 2017 population — labels from official train_v2.csv if available,
#           else fallback to FE-derived labels
#   Features: always computed from our FE pipeline (full tx history <= cutoff)

def build_snapshot_with_official_labels(
    month: pd.Period,
    transactions: pd.DataFrame,
    members: pd.DataFrame,
    official_labels=None,
    grace_days: int = 30,
    split_name: str = "train",
):
    """
    Xây dựng snapshot cho một tháng:
    1. Luôn dùng FE pipeline để tính features (full history <= cutoff)
    2. Nếu có official_labels: dùng nhãn chính thức từ competition
       Nếu không: fallback sang FE-derived labels (build_labels_vectorized)
    3. Merge labels + features trên msno
    """
    cutoff = month_end(month)
    start  = time.time()

    if official_labels is not None:
        # ── Option B: official labels ──────────────────────────────────────
        # Population = users in official label file
        pop_users   = set(official_labels["msno"].astype(str).tolist())
        labels_snap = official_labels[["msno", "is_churn"]].copy()
        labels_snap["snapshot_date"] = cutoff
        labels_snap["last_expire"]   = pd.NaT      # not available from official file
        labels_snap["label_source"]  = "official"
        label_type = "official"
    else:
        # ── Fallback: FE-derived labels ────────────────────────────────────
        labels_snap = build_labels_vectorized(
            transactions, month, grace_days=grace_days, debug=False
        )
        if len(labels_snap) == 0:
            return None
        pop_users  = set(labels_snap["msno"].astype(str).tolist())
        label_type = "fe_derived"

    # ── Features: always from FE pipeline ──────────────────────────────────
    feat_snap = build_core_feature_snapshot(
        cutoff, transactions, members,
        population_users=pop_users,
        # Pass logs only if coverage >= 20% — else logs add zero-inflated noise
        logs_df=user_logs_hist if USE_LOGS else None,
    )

    # ── Merge labels + features ────────────────────────────────────────────
    snap = labels_snap.merge(feat_snap, on=["msno", "snapshot_date"], how="left")
    snap["dataset_split"] = split_name

    elapsed    = time.time() - start
    churn_rate = float(snap["is_churn"].mean()) if "is_churn" in snap.columns else 0
    print(
        f"  [{split_name}] {month}: pop={len(pop_users):,} | "
        f"labels={len(labels_snap):,} | snap={snap.shape[0]:,} | "
        f"churn={churn_rate:.2%} | source={label_type} | {elapsed:.2f}s"
    )
    return snap


# ── Zero-transaction user diagnostic ────────────────────────────────────
# Users in official labels with no transactions in clean_transactions_core
# These will have all tx features = 0 — verify this is acceptable
if official_train_labels is not None:
    train_msnos   = set(official_train_labels["msno"].astype(str))
    tx_msnos      = set(transactions["msno"].astype(str))
    no_tx_users   = train_msnos - tx_msnos
    pct_no_tx     = len(no_tx_users) / len(train_msnos)
    print(f"Train users with NO transactions : {len(no_tx_users):,} ({pct_no_tx:.1%})")
    print(f"Train users WITH transactions    : {len(train_msnos)-len(no_tx_users):,} ({1-pct_no_tx:.1%})")
    if pct_no_tx > 0.05:
        print("  ⚠️  >5% users have zero tx features — check transaction coverage")
    else:
        print("  ✅ <5% zero-tx users — acceptable")

print("\n" + "="*90)
print("BUILDING TRAIN SNAPSHOT — January 2017")
print("="*90)

pipeline_start = time.time()

train_snap = build_snapshot_with_official_labels(
    TRAIN_MONTHS[0],
    transactions, members,
    official_labels=official_train_labels,  # None if not found → fallback
    grace_days=GRACE_DAYS,
    split_name="train",
)
if train_snap is None:
    raise ValueError("✗ ERROR: Train snapshot is empty.")
train_dataset = train_snap

print("\n" + "="*90)
print("BUILDING VALIDATION SNAPSHOT — February 2017")
print("="*90)

val_snap = build_snapshot_with_official_labels(
    VAL_MONTHS[0],
    transactions, members,
    official_labels=official_val_labels,    # None if not found → fallback
    grace_days=GRACE_DAYS,
    split_name="validation",
)
if val_snap is None:
    raise ValueError("✗ ERROR: Validation snapshot is empty.")
val_dataset = val_snap

master_multi  = pd.concat([train_dataset, val_dataset], axis=0, ignore_index=True)
combined_time = time.time() - pipeline_start
gc.collect()

print(f"\n{'='*90}")
print(f"✓ TRAIN+VAL COMPLETE: {master_multi.shape[0]:,} rows × {master_multi.shape[1]:,} cols")
print(f"  Time: {combined_time:.2f}s")
print(f"{'='*90}")
print("\nRows per snapshot:")
print(master_multi.groupby("snapshot_date").size())
print("\nChurn rate per snapshot:")
print(master_multi.groupby("snapshot_date")["is_churn"].mean())
print("\nRows by split:")
print(master_multi.groupby("dataset_split").size())
print("\nLabel source breakdown:")
print(master_multi.groupby(["dataset_split", "label_source"]).size())
print("\nFirst 5 rows:")
display(master_multi.head())


Train users with NO transactions : 44,584 (4.5%)
Train users WITH transactions    : 948,347 (95.5%)
  ✅ <5% zero-tx users — acceptable

BUILDING TRAIN SNAPSHOT — January 2017
  [train] 2017-01: pop=992,931 | labels=992,931 | snap=992,931 | churn=6.39% | source=official | 22.01s

BUILDING VALIDATION SNAPSHOT — February 2017
  [validation] 2017-02: pop=970,960 | labels=970,960 | snap=970,960 | churn=8.99% | source=official | 22.19s

✓ TRAIN+VAL COMPLETE: 1,963,891 rows × 60 cols
  Time: 44.79s

Rows per snapshot:
snapshot_date
2017-01-31    992931
2017-02-28    970960
dtype: int64

Churn rate per snapshot:
snapshot_date
2017-01-31    0.063923
2017-02-28    0.089942
Name: is_churn, dtype: float64

Rows by split:
dataset_split
train         992931
validation    970960
dtype: int64

Label source breakdown:
dataset_split  label_source
train          official        992931
validation     official        970960
dtype: int64

First 5 rows:


,msno,is_churn,snapshot_date,last_expire,label_source,bd,bd_missing,city,city_missing,gender,gender_missing,registered_via,registered_via_missing,days_since_reg,registration_date_abs,n_txns,auto_renew_rate,last_is_auto_renew,avg_plan_days,plan_days_std,plan_change_flag,last_plan_days,first_plan_days,max_plan_days,total_amount_paid,avg_amount_paid,max_amount_paid,zero_paid_rate,avg_discount_rate,share_30d,share_90d,days_since_last_txn,tenure_days,cancel_rate,cancel_count,last_is_cancel,cancel_in_last_month,avg_cancel_per_month,cancel_to_plandays_ratio,active_no_cancel_flag,days_since_last_cancel,has_ever_cancelled,payment_method_nunique,days_last_txn_to_expire,recency_to_plan_ratio,spend_rate_per_day,total_secs_played,num_logins,secs_played_14d,logins_14d,secs_played_30d,logins_30d,logins_90d,secs_played_60_30d,delta_secs_30d_vs_prior,days_since_last_login,max_secs_single_day,days_since_significant_usage,has_log_history,dataset_split
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1,2017-01-31,NaT,official,36.0,0.0,18,0.0,female,0.0,9,0.0,4323.0,20050406.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1,2017-01-31,NaT,official,38.0,0.0,10,0.0,male,0.0,9,0.0,4323.0,20050407.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1,2017-01-31,NaT,official,27.0,0.0,11,0.0,female,0.0,9,0.0,4140.0,20051016.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1,2017-01-31,NaT,official,23.0,0.0,13,0.0,female,0.0,9,0.0,4109.0,20051102.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1,2017-01-31,NaT,official,27.0,0.0,3,0.0,male,0.0,9,0.0,4079.0,20051228.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train


## March 2017 Inference Snapshot

The inference snapshot has no labels. Population is taken from `sample_submission_v2.csv` (907,471 users) rather than from FE effective state, because the FE population filter only produced 32,808 users — too restrictive relative to the competition's target population.

Features are computed with cutoff = 2017-03-31, meaning all transaction and log history up to end of March is used. This differs from train/val: inference additionally includes March log data from `user_logs_march`.

The `snapshot_date` column is set to 2017-03-31 across all inference rows to ensure column alignment with train/val when the model predicts.


In [33]:
# ===== 10. Build March inference snapshot =====
print("\n" + "="*90)
print("BUILDING MARCH INFERENCE SNAPSHOT")
print("="*90)

inf_start = time.time()

# Population: users whose effective state expires in 2017-03
# Use build_labels_vectorized with a dummy future window (no labels needed)
# Use official submission population (907K) if available
# else fall back to FE-derived effective state (32K)
if submission_df is not None:
    march_population = set(submission_df["msno"].astype(str).tolist())
    print(f"March population from sample_submission_v2: {len(march_population):,} ✅")
else:
    march_pop_df = build_labels_vectorized(
        transactions, INF_MONTH, grace_days=GRACE_DAYS, debug=True
    )
    march_population = set(march_pop_df["msno"].astype(str).tolist())
    print(f"March population from FE (fallback): {len(march_population):,}")

inference_snapshot = build_core_feature_snapshot(
    inf_cutoff, transactions, members,
    population_users=march_population,
    logs_df=user_logs_march if USE_LOGS else None,
)
# Add snapshot_date alignment
inference_snapshot["snapshot_date"] = inf_cutoff

inf_time = time.time() - inf_start
print(f"inference_snapshot shape  : {inference_snapshot.shape}")
print(f"inference unique users    : {inference_snapshot['msno'].nunique():,}")
print(f"inference built in        : {inf_time:.2f}s")
print("\nFirst 5 rows:")
display(inference_snapshot.head())



BUILDING MARCH INFERENCE SNAPSHOT
March population from sample_submission_v2: 907,471 ✅
inference_snapshot shape  : (907471, 56)
inference unique users    : 907,471
inference built in        : 39.77s

First 5 rows:


,msno,snapshot_date,bd,bd_missing,city,city_missing,gender,gender_missing,registered_via,registered_via_missing,days_since_reg,registration_date_abs,n_txns,auto_renew_rate,last_is_auto_renew,avg_plan_days,plan_days_std,plan_change_flag,last_plan_days,first_plan_days,max_plan_days,total_amount_paid,avg_amount_paid,max_amount_paid,zero_paid_rate,avg_discount_rate,share_30d,share_90d,days_since_last_txn,tenure_days,cancel_rate,cancel_count,last_is_cancel,cancel_in_last_month,avg_cancel_per_month,cancel_to_plandays_ratio,active_no_cancel_flag,days_since_last_cancel,has_ever_cancelled,payment_method_nunique,days_last_txn_to_expire,recency_to_plan_ratio,spend_rate_per_day,total_secs_played,num_logins,secs_played_14d,logins_14d,secs_played_30d,logins_30d,logins_90d,secs_played_60_30d,delta_secs_30d_vs_prior,days_since_last_login,max_secs_single_day,days_since_significant_usage,has_log_history
0,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,2017-03-31,27.0,1.0,1,0.0,Unknown,1.0,7,0.0,2038.0,20110914.0,1.0,1.0,1.0,30.0,0.0,0.0,30.0,30.0,30.0,129.0,129.0,129.0,0.0,0.0,1.0,1.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9999.0,0.0,1.0,32.0,0.133333,32.250000,0.000,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0
1,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,2017-03-31,32.0,0.0,6,0.0,female,0.0,9,0.0,2038.0,20110915.0,1.0,1.0,1.0,30.0,0.0,0.0,30.0,30.0,30.0,149.0,149.0,149.0,0.0,0.0,1.0,1.0,17.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9999.0,0.0,1.0,30.0,0.566667,8.764706,1970.921,1.0,1970.921,1.0,1970.921,1.0,1.0,0.0,65.697367,13.0,1970.921,13.0,1
2,I0yFvqMoNkM8ZNHb617e1RBzIS/YRKemHO7Wj13EtA0=,2017-03-31,63.0,0.0,13,0.0,male,0.0,9,0.0,2038.0,20110918.0,1.0,1.0,1.0,30.0,0.0,0.0,30.0,30.0,30.0,149.0,149.0,149.0,0.0,0.0,1.0,1.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9999.0,0.0,1.0,30.0,0.133333,37.250000,0.000,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0
3,OoDwiKZM+ZGr9P3fRivavgOtglTEaNfWJO4KaJcTTts=,2017-03-31,27.0,1.0,1,0.0,Unknown,1.0,7,0.0,2038.0,20110918.0,1.0,1.0,1.0,30.0,0.0,0.0,30.0,30.0,30.0,149.0,149.0,149.0,0.0,0.0,1.0,1.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9999.0,0.0,1.0,32.0,0.166667,29.800000,0.000,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0
4,4De1jAxNRABoyRBDZ82U0yEmzYkqeOugRGVNIf92Xb8=,2017-03-31,28.0,0.0,4,0.0,female,0.0,9,0.0,2038.0,20110920.0,1.0,1.0,1.0,30.0,0.0,0.0,30.0,30.0,30.0,180.0,180.0,180.0,0.0,0.0,1.0,1.0,28.0,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9999.0,0.0,1.0,30.0,0.933333,6.428571,0.000,0.0,0.000,0.0,0.000,0.0,0.0,0.0,0.000000,0.0,0.000,0.0,0


## Auxiliary Logs Branch

Historical logs and March logs are retained as supplementary information. They are not required inputs for the core model but allow behavioral analysis of specific user segments.

Coverage check results:
- Historical logs cover 1.1% of Jan 2017 train population
- March logs cover 22.9% of train population
- Combined (hist + march): 23.7% coverage

The low coverage is a property of the public Kaggle dataset — the competition only provided logs for a subset of users. For the remaining 76%, log features are zero-filled, which is a valid signal in its own right.


In [34]:

# ===== 11. Auxiliary logs info =====
if user_logs_hist is not None:
    print("Historical logs users:", user_logs_hist["msno"].nunique())
    print("Historical logs range:", user_logs_hist["date"].min(), "->", user_logs_hist["date"].max())
if user_logs_march is not None:
    print("March logs users:", user_logs_march["msno"].nunique())
    print("March logs range:", user_logs_march["date"].min(), "->", user_logs_march["date"].max())


Historical logs users: 22443
Historical logs range: 2015-01-01 00:00:00 -> 2017-02-28 00:00:00
March logs users: 316345
March logs range: 2017-03-01 00:00:00 -> 2017-03-31 00:00:00


## Export

Seven files are exported after Feature Engineering completes:

| File | Contents | Used for |
|---|---|---|
| `final_df.parquet` | Train + val, all columns | Analysis, debugging |
| `master_table.parquet` | Copy of final_df | Backup |
| `master_model_table.parquet` | Train + val, meta columns separated | Modeling |
| `processed_master.parquet` | Copy of master_model_table | Backup |
| `inference_snapshot.parquet` | March 2017 features | Prediction |
| `snapshot_summary.csv` | Churn rate statistics by month | Sanity check |
| `feature_engineering_metadata_v7.json` | Run metadata | Reproducibility |

Column structure in `master_model_table`:
- Meta columns (6): `msno`, `snapshot_date`, `last_expire`, `is_churn`, `label_source`, `dataset_split`
- Feature columns (53): all features to be passed to the model


In [35]:

# ===== 12. Export =====
master_export = master_multi.copy()

for col in master_export.columns:
    dtype_str = str(master_export[col].dtype)
    if dtype_str in ["category", "object", "string"]:
        master_export[col] = master_export[col].astype("string")
    elif "datetime" in dtype_str:
        master_export[col] = pd.to_datetime(master_export[col], errors="coerce")
    elif dtype_str == "bool":
        master_export[col] = master_export[col].astype("int8")

sort_cols = [c for c in ["snapshot_date", "msno"] if c in master_export.columns]
if len(sort_cols) > 0:
    master_export = master_export.sort_values(sort_cols).reset_index(drop=True)

meta_cols = [c for c in ["msno", "snapshot_date", "last_expire",
             "is_churn", "label_source", "dataset_split"]
             if c in master_export.columns]
feature_cols = [c for c in master_export.columns if c not in meta_cols]

master_model_table = master_export[meta_cols + feature_cols].copy()

master_export.to_parquet(DATA_DIR / "final_df.parquet", index=False)
master_export.to_parquet(DATA_DIR / "master_table.parquet", index=False)
master_model_table.to_parquet(DATA_DIR / "master_model_table.parquet", index=False)
master_model_table.to_parquet(DATA_DIR / "processed_master.parquet", index=False)

snapshot_summary = (
    master_export.groupby(["dataset_split", "snapshot_date"])
    .agg(
        n_rows=("is_churn", "size"),
        churn_rate=("is_churn", "mean"),
        n_users=("msno", "nunique")
    )
    .reset_index()
)
snapshot_summary.to_csv(DATA_DIR / "snapshot_summary.csv", index=False)

run_metadata = {
    "n_rows": int(master_export.shape[0]),
    "n_cols": int(master_export.shape[1]),
    "n_snapshots": int(master_export["snapshot_date"].nunique()),
    "feature_count": int(len(feature_cols)),
    "feature_names": feature_cols,
    "setup": {
        "train_months": [str(m) for m in TRAIN_MONTHS],
        "validation_months": [str(m) for m in VAL_MONTHS],
        "inference_month": str(INF_MONTH),
        "grace_days": GRACE_DAYS,
        "population_rule": "effective-state inspired via Scala-aligned state selection",
        "label_rule": "no valid renewal within 30 days after expiry",
        "feature_rule": "all prior activity before month-end cutoff"
    },
    "core_model_uses_logs": False,
    "notes": [
        "v7: vectorized label builder, Scala-aligned effective state",
        "Bryan-style split: Train=Jan 2017, Val=Feb 2017, Inf=Mar 2017",
        "Option B: official train.csv/train_v2.csv labels used if available",
        "Features always from FE pipeline regardless of label source",
        "label_source=official|fe_derived|no_future_data|beyond_window"
    ]
}

with open(DATA_DIR / "feature_engineering_metadata_v7.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2, default=str)

inf_export = inference_snapshot.copy()
for col in inf_export.columns:
    if str(inf_export[col].dtype) in ["category", "object", "string"]:
        inf_export[col] = inf_export[col].astype("string")
inf_export.to_parquet(DATA_DIR / "inference_snapshot.parquet", index=False)

print("Saved:")
print(DATA_DIR / "final_df.parquet")
print(DATA_DIR / "master_table.parquet")
print(DATA_DIR / "master_model_table.parquet")
print(DATA_DIR / "processed_master.parquet")
print(DATA_DIR / "snapshot_summary.csv")
print(DATA_DIR / "feature_engineering_metadata_v7.json")
print(DATA_DIR / "inference_snapshot.parquet")


Saved:
Data\final_df.parquet
Data\master_table.parquet
Data\master_model_table.parquet
Data\processed_master.parquet
Data\snapshot_summary.csv
Data\feature_engineering_metadata_v7.json
Data\inference_snapshot.parquet


In [36]:

# ===== 13. Final sanity =====
print("master_export shape:", master_export.shape)
print("snapshot_date unique:", master_export["snapshot_date"].nunique())

print("\nRows per snapshot:")
display(master_export.groupby("snapshot_date").size())

print("\nChurn rate per snapshot:")
display(master_export.groupby("snapshot_date")["is_churn"].mean())

print("\nRows by split:")
display(master_export.groupby("dataset_split").size())

print("\nLabel source breakdown:")
display(master_export.groupby(["dataset_split", "label_source"]).size())

print("\nFeature columns:")
print(feature_cols)

# Verify no meta cols leaked into features
meta_cols_check = ["msno", "snapshot_date", "last_expire", "is_churn",
                   "label_source", "dataset_split"]
leaked = [c for c in meta_cols_check if c in feature_cols]
print(f"\nMeta cols leaked into features: {leaked if leaked else 'NONE ✅'}")
print(f"Total features: {len(feature_cols)}")

print("\nReminder:")
print("  - If label_source='official': labels from train.csv/train_v2.csv (ground truth)")
print("  - If label_source='fe_derived': labels from FE pipeline (fallback)")
print("  - Filter label_source != 'beyond_window' before training")
print("  - Exclude meta cols from feature matrix X")


master_export shape: (1963891, 60)
snapshot_date unique: 2

Rows per snapshot:


snapshot_date
2017-01-31    992931
2017-02-28    970960
dtype: int64


Churn rate per snapshot:


snapshot_date
2017-01-31    0.063923
2017-02-28    0.089942
Name: is_churn, dtype: float64


Rows by split:


dataset_split
train         992931
validation    970960
dtype: int64


Label source breakdown:


dataset_split  label_source
train          official        992931
validation     official        970960
dtype: int64


Feature columns:
['bd', 'bd_missing', 'city', 'city_missing', 'gender', 'gender_missing', 'registered_via', 'registered_via_missing', 'days_since_reg', 'registration_date_abs', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'last_plan_days', 'first_plan_days', 'max_plan_days', 'total_amount_paid', 'avg_amount_paid', 'max_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'last_is_cancel', 'cancel_in_last_month', 'avg_cancel_per_month', 'cancel_to_plandays_ratio', 'active_no_cancel_flag', 'days_since_last_cancel', 'has_ever_cancelled', 'payment_method_nunique', 'days_last_txn_to_expire', 'recency_to_plan_ratio', 'spend_rate_per_day', 'total_secs_played', 'num_logins', 'secs_played_14d', 'logins_14d', 'secs_played_30d', 'logins_30d', 'logins_90d', 'secs_played_60_30d', 'delta_secs_30d_vs_prior', 'days_since_last_login', 'max_secs_sing